# Socio-Spatial Analysis of Housing Prices in Paris

## 1. Introduction

In this project, we aim to model and analyze housing price variations in Paris using the French DVF (Demande de Valeurs Foncières) dataset.

We focus on predicting the price per square meter (€/m²) of residential properties, using a set of structural, spatial, and temporal features derived from transaction data. Given the complexity of real estate markets and the limitations of available features in the DVF dataset, this project adopts a baseline machine learning approach to evaluate how much of the price variation can be explained by basic observable characteristics.

In particular, we construct a clean subset of apartment transactions in Paris, engineer relevant features such as location (arrondissement), transaction time, and property characteristics, and apply a linear regression model as a baseline. The results are then analyzed to assess the limitations of the model and the data, and to identify directions for improvement.

In [207]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Data Description & Preprocessing  

### 2.1 Data Source (DVF)

In [208]:
df = pd.read_csv("dvf.csv")

/var/folders/c8/vcr8t_yn0wq2rqnv1b7cqrvr0000gn/T/ipykernel_3577/681951132.py:1: DtypeWarning: Columns (2,4,5,8,9,10,12,13,14,16,17,18,19,20,21,22,23,24,25,26,27,28,29,31,32,35,36,37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dvf.csv")


In [209]:
df.columns

Index(['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation',
       'valeur_fonciere', 'adresse_numero', 'adresse_suffixe',
       'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune',
       'nom_commune', 'code_departement', 'ancien_code_commune',
       'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle',
       'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero',
       'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez',
       'lot4_numero', 'lot4_surface_carrez', 'lot5_numero',
       'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local',
       'surface_reelle_bati', 'nombre_pieces_principales',
       'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale',
       'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude',
       'section_prefixe'],
      dtype='object')

### 2.2 Data Cleaning

In [210]:
df = df.reset_index(drop=True)
data = df[[
    "id_mutation",
    "nature_mutation",
    "date_mutation",
    "valeur_fonciere",
    "code_postal",
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "longitude", 
    "latitude",
    "nombre_lots"
]].copy()
data.head()

,id_mutation,nature_mutation,date_mutation,valeur_fonciere,code_postal,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,nombre_lots
0,2024-1190360,Vente,2024-01-04,1042000.0,75020.0,Dépendance,NaN,0.0,2.405228,48.868216,2
1,2024-1190360,Vente,2024-01-04,1042000.0,75020.0,Dépendance,NaN,0.0,2.405228,48.868216,1
2,2024-1190360,Vente,2024-01-04,1042000.0,75020.0,Appartement,86.0,4.0,2.405228,48.868216,2
3,2024-1190360,Vente,2024-01-04,1042000.0,75020.0,Dépendance,NaN,0.0,2.405228,48.868216,1
4,2024-1190360,Vente,2024-01-04,1042000.0,75020.0,Dépendance,NaN,0.0,2.405228,48.868216,2


In [211]:
# convert numeric columns
data["valeur_fonciere"] = pd.to_numeric(data["valeur_fonciere"], errors="coerce")
data["surface_reelle_bati"] = pd.to_numeric(data["surface_reelle_bati"], errors="coerce")
data["nombre_pieces_principales"] = pd.to_numeric(data["nombre_pieces_principales"], errors="coerce")
data["nombre_lots"] = pd.to_numeric(data["nombre_lots"], errors="coerce")
data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")

# filter
data = data[
    (data["type_local"] == "Appartement") &
    (data["nature_mutation"] == "Vente") &
    (data["valeur_fonciere"] > 0) &
    (data["surface_reelle_bati"] > 0) &
    (data["nombre_pieces_principales"] > 0) &
    (data["nombre_lots"] == 1)
].copy()

data = data.dropna(subset=[
    "valeur_fonciere",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "date_mutation",
    "code_postal",
    "longitude", 
    "latitude"
])
data = data[data["nombre_lots"] == 1].copy()
data = data.drop_duplicates(subset="id_mutation").copy()
data = data.reset_index(drop=True)
data.head()

,id_mutation,nature_mutation,date_mutation,valeur_fonciere,code_postal,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,nombre_lots
0,2024-1190362,Vente,2024-01-03,426471.0,75010.0,Appartement,42.0,1.0,2.370282,48.871532,1.0
1,2024-1190363,Vente,2024-01-08,778000.0,75003.0,Appartement,63.0,3.0,2.360510,48.859577,1.0
2,2024-1190364,Vente,2024-01-05,630000.0,75019.0,Appartement,66.0,2.0,2.369590,48.884423,1.0
3,2024-1190365,Vente,2024-01-05,173025.0,75020.0,Appartement,17.0,1.0,2.405022,48.867372,1.0
4,2024-1190367,Vente,2024-01-03,392000.0,75019.0,Appartement,40.0,2.0,2.391306,48.878743,1.0


In [212]:
data["id_mutation"].duplicated().sum()

np.int64(0)

### 2.3 Construction of Price per m²

In [213]:
# target
data["prix_m2"] = (data["valeur_fonciere"] / data["surface_reelle_bati"]).round(2)
data.head()

,id_mutation,nature_mutation,date_mutation,valeur_fonciere,code_postal,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,nombre_lots,prix_m2
0,2024-1190362,Vente,2024-01-03,426471.0,75010.0,Appartement,42.0,1.0,2.370282,48.871532,1.0,10154.07
1,2024-1190363,Vente,2024-01-08,778000.0,75003.0,Appartement,63.0,3.0,2.360510,48.859577,1.0,12349.21
2,2024-1190364,Vente,2024-01-05,630000.0,75019.0,Appartement,66.0,2.0,2.369590,48.884423,1.0,9545.45
3,2024-1190365,Vente,2024-01-05,173025.0,75020.0,Appartement,17.0,1.0,2.405022,48.867372,1.0,10177.94
4,2024-1190367,Vente,2024-01-03,392000.0,75019.0,Appartement,40.0,2.0,2.391306,48.878743,1.0,9800.00


### 2.4 Remove Outliers

In [214]:
# remove outliers
data["log_prix_m2"] = np.log(data["prix_m2"])
q1 = data["log_prix_m2"].quantile(0.01)
q99 = data["log_prix_m2"].quantile(0.99)

data = data[
    (data["log_prix_m2"] >= q1) &
    (data["log_prix_m2"] <= q99) &
    (data["surface_reelle_bati"] >= 10) &
    (data["surface_reelle_bati"] <= 250)
].copy()
data.head()

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,id_mutation,nature_mutation,date_mutation,valeur_fonciere,code_postal,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,nombre_lots,prix_m2,log_prix_m2
0,2024-1190362,Vente,2024-01-03,426471.0,75010.0,Appartement,42.0,1.0,2.370282,48.871532,1.0,10154.07,9.225630
1,2024-1190363,Vente,2024-01-08,778000.0,75003.0,Appartement,63.0,3.0,2.360510,48.859577,1.0,12349.21,9.421347
2,2024-1190364,Vente,2024-01-05,630000.0,75019.0,Appartement,66.0,2.0,2.369590,48.884423,1.0,9545.45,9.163820
3,2024-1190365,Vente,2024-01-05,173025.0,75020.0,Appartement,17.0,1.0,2.405022,48.867372,1.0,10177.94,9.227978
4,2024-1190367,Vente,2024-01-03,392000.0,75019.0,Appartement,40.0,2.0,2.391306,48.878743,1.0,9800.00,9.190138


### 2.5 Prepare Features

In [ ]:
data["date_mutation"] = pd.to_datetime(data["date_mutation"], errors="coerce")
data["code_postal"] = pd.to_numeric(data["code_postal"], errors="coerce")
data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")

data = data.dropna(subset=["date_mutation", "code_postal", "longitude", "latitude"])

data["year"] = data["date_mutation"].dt.year
data["month"] = data["date_mutation"].dt.month
data["arrondissement"] = (data["code_postal"] - 75000).astype(int)

data = data.drop(columns=["date_mutation", "code_postal"])
data.head()

,id_mutation,nature_mutation,valeur_fonciere,type_local,surface_reelle_bati,nombre_pieces_principales,longitude,latitude,nombre_lots,prix_m2,log_prix_m2,year,month,arrondissement
0,2024-1190362,Vente,426471.0,Appartement,42.0,1.0,2.370282,48.871532,1.0,10154.07,9.225630,2024,1,10
1,2024-1190363,Vente,778000.0,Appartement,63.0,3.0,2.360510,48.859577,1.0,12349.21,9.421347,2024,1,3
2,2024-1190364,Vente,630000.0,Appartement,66.0,2.0,2.369590,48.884423,1.0,9545.45,9.163820,2024,1,19
3,2024-1190365,Vente,173025.0,Appartement,17.0,1.0,2.405022,48.867372,1.0,10177.94,9.227978,2024,1,20
4,2024-1190367,Vente,392000.0,Appartement,40.0,2.0,2.391306,48.878743,1.0,9800.00,9.190138,2024,1,19


In [ ]:
# Choose features
features = [
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "arrondissement",
    "month",
    "year",
    "longitude", 
    "latitude"
]

In [217]:
X = data[features]
y = data["log_prix_m2"]

## 3. Modeling

In [218]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [219]:
categorical_features = ["arrondissement", "month"]
numeric_features = ["surface_reelle_bati", "nombre_pieces_principales", "year", "longitude","latitude"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['arrondissement', 'month']),
                                                 ('num', 'passthrough',
                                                  ['surface_reelle_bati',
                                                   'nombre_pieces_principales',
                                                   'year', 'longitude',
                                                   'latitude'])])),
                ('regressor', LinearRegression())])

In [220]:
y_pred = model.predict(X_test)

## 4. Results & Evaluation 

### 4.1 Performance Metrics (R², RMSE, MAE)

In [221]:
print("R²:", r2_score(y_test, y_pred))
print("RMSE (log):", np.sqrt(mean_squared_error(y_test, y_pred)))

R²: 0.019505718146752615
RMSE (log): 0.9357287463141497


In [222]:
y_pred_real = np.exp(y_pred)
y_test_real = np.exp(y_test)

print("MAE (€ / m²):", mean_absolute_error(y_test_real, y_pred_real))

MAE (€ / m²): 3677.1586867695005


### 4.2 Discussion of Results

The linear regression baseline achieves very limited explanatory power, as reflected by a low R² score. This indicates that the selected features are insufficient to capture the complexity of housing price variation in Paris.

Several factors may explain this result. First, the DVF dataset lacks important property-level attributes such as building age, condition, and amenities, which are known to significantly influence prices. Second, spatial information at the arrondissement level is too coarse to capture fine-grained differences within neighborhoods. Finally, the relationship between features and housing prices is likely nonlinear, which cannot be effectively modeled by a simple linear approach.

## 5. Conclusion

In this project, we developed a baseline model to analyze housing price variation in Paris using the DVF dataset. Despite careful data cleaning and feature engineering, the model achieved limited predictive performance, highlighting the challenges of modeling real estate prices using simplified features.

This study demonstrates that housing prices in Paris are driven by complex spatial and structural factors that are not fully captured by basic administrative data. In particular, the results emphasize the importance of fine-grained spatial information and richer property attributes.

More broadly, this project illustrates both the usefulness and the limitations of open transaction data for urban analysis. Future work could improve predictive performance by incorporating more detailed spatial features and applying more flexible machine learning models.